In [1]:
!pip install fastapi uvicorn nest_asyncio
!pip install pyngrok
!apt-get update && apt-get install -y build-essential cmake
!CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install --verbose --no-cache-dir llama-cpp-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 103.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjit

In [2]:
from pyngrok import ngrok
from huggingface_hub import login

#ngrok 토큰
ngrok.set_auth_token("")
#huggingface 토큰
login("")

In [ ]:
!huggingface-cli download MLP-KTLim/llama-3-Korean-Bllossom-8B-gguf-Q4_K_M \
--include "*.gguf" \
--local-dir ./model

In [ ]:
from fastapi import FastAPI, Request
from llama_cpp import Llama
from pyngrok import ngrok
import nest_asyncio
import uvicorn
import os
import asyncio

# 모델 경로
MODEL_PATH = "./model/llama-3-Korean-Bllossom-8B-Q4_K_M.gguf"

# llama.cpp 모델 로딩
llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=35,    # GPU에서 실행할 레이어 수 (T4에서 40 권장)
    n_ctx=2048,
    verbose=False
)

# FastAPI 초기화
app = FastAPI()

@app.post("/inference")
async def generate_answer(request: Request):
    body = await request.json()
    prompt = body.get("prompt", "")

    loop = asyncio.get_event_loop()
    output = await loop.run_in_executor(
        None,
        lambda: llm(
            prompt=prompt,
            max_tokens=512,
            temperature=0.7,
            top_p=0.9,
            stop=["</s>"]
        )
    )

    answer_text = output["choices"][0]["text"].strip()
    return {"result": answer_text}

@app.get("/")
async def root():
    return {"message": "FastAPI + llama.cpp 서버가 실행 중입니다."}

# ngrok 설정
public_url = ngrok.connect(7860)
print("외부 접속 주소:", public_url)

nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=7860)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/710 [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/172 [00:00<?, ?B/s]

외부 접속 주소: NgrokTunnel: "https://f6fd980ae8c0.ngrok-free.app" -> "http://localhost:7860"


INFO:     Started server process [828]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:7860 (Press CTRL+C to quit)


INFO:     221.148.97.238:0 - "GET / HTTP/1.1" 200 OK
INFO:     221.148.97.238:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     221.148.97.238:0 - "POST /inference HTTP/1.1" 200 OK
